# Participación Marzo 19

In [18]:
# Imports

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

## Cargar y Particionar Dataset

In [19]:
# Cargar conjunto de datos y particionar (80/10/10)

x, y = fetch_california_housing(return_X_y=True, as_frame=True)

x_train, x_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_testval, y_testval, test_size=0.5, random_state=42)

In [20]:
# Obtener índices de columnas que se transformarán (se usó apoyo de la IA para esta parte)

cols_to_transform = [col for col in x.columns if col not in ['Latitude', 'Longitude']]
cols_passthrough = ['Latitude', 'Longitude']

cols = list(x.columns)

cols_to_transform_idx = [cols.index(c) for c in cols_to_transform]
cols_passthrough_idx = [cols.index(c) for c in cols_passthrough]

cols_to_transform_idx = torch.tensor(cols_to_transform_idx, dtype=torch.long)
cols_passthrough_idx = torch.tensor(cols_passthrough_idx, dtype=torch.long)

print(cols_to_transform_idx)
print(cols_passthrough_idx)

tensor([0, 1, 2, 3, 4, 5])
tensor([6, 7])


## Capa de Preprocesamiento (Eliminación de Outliers y Standard Scaling)

In [21]:
# Definir variables para eliminar outliers

Q1 = x_train[cols_to_transform].quantile(0.25)
Q3 = x_train[cols_to_transform].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

lower_bound = torch.tensor(lower_bound.values, dtype=torch.float32)
upper_bound = torch.tensor(upper_bound.values, dtype=torch.float32)


# Definir variables para aplicar escalamiento

mean = x_train[cols_to_transform].mean()
std = x_train[cols_to_transform].std()

mean = torch.tensor(mean.values, dtype=torch.float32)
std = torch.tensor(std.values, dtype=torch.float32)

# Capa de preprocesamiento

class PreprocessingLayer(nn.Module):
    def __init__(self, lower_bound: torch.Tensor, upper_bound: torch.Tensor, cols_to_transform_idx: torch.Tensor, cols_passthrough_idx: torch.Tensor, mean: torch.Tensor, std: torch.Tensor):
        super().__init__()
        self.register_buffer('lower_bound', lower_bound)
        self.register_buffer('upper_bound', upper_bound)
        self.register_buffer('cols_to_transform_idx', cols_to_transform_idx)
        self.register_buffer('cols_passthrough_idx', cols_passthrough_idx)
        self.register_buffer('mean', mean)
        self.register_buffer('std', std)
    
    def forward(self, X: torch.Tensor) -> torch.Tensor:
        x_transform = X[:, self.cols_to_transform_idx]
        x_pass = X[:, self.cols_passthrough_idx]

        x_transform = torch.clamp(x_transform, self.lower_bound, self.upper_bound)

        x_transform = (x_transform - self.mean) / (self.std + 1e-8)

        x_out = torch.cat([x_transform, x_pass], dim=1)

        return x_out


## Definir y Entrenar 3 modelos

In [22]:
# Transformar datos a tensores para entrenamiento y predicciones (se usó apoyo de la IA)
x_train = torch.tensor(x_train.values, dtype=torch.float32)
x_val   = torch.tensor(x_val.values, dtype=torch.float32)
x_test  = torch.tensor(x_test.values, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_val   = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)
y_test  = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# Aplicar DataLoader
dataset = TensorDataset(x_train, y_train)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Definir función de pérdida
loss_fn = nn.MSELoss()

In [23]:
# Definir Modelo 1
model_1 = nn.Sequential(
    PreprocessingLayer(
        lower_bound, upper_bound,
        cols_to_transform_idx, cols_passthrough_idx,
        mean, std
    ),
    nn.Linear(8, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

optimizer_1 = torch.optim.SGD(model_1.parameters(), lr=0.01)
num_epochs_1 = 50


# Definir Modelo 2
model_2 = nn.Sequential(
    PreprocessingLayer(
        lower_bound, upper_bound,
        cols_to_transform_idx, cols_passthrough_idx,
        mean, std
    ),
    nn.Linear(8, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

optimizer_2 = torch.optim.Adam(model_2.parameters(), lr=0.001)
num_epochs_2 = 100

# Definir Modelo 3
# Modelo 3
model_3 = nn.Sequential(
    PreprocessingLayer(
        lower_bound, upper_bound,
        cols_to_transform_idx, cols_passthrough_idx,
        mean, std
    ),
    nn.Linear(8, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

optimizer_3 = torch.optim.Adam(model_3.parameters(), lr=0.0005)
num_epochs_3 = 100


In [24]:
# Función para entrenar los modelos
def train_model(model, num_epochs, optimizer, loss_fn):
    for epoch in range(num_epochs):
        model.train()

        for X_batch, y_batch in tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            predictions = model(X_batch)
            loss = loss_fn(predictions, y_batch)
            optimizer.zero_grad() 
            loss.backward() 
            optimizer.step()


In [25]:
# Entrenar Modelo 1
train_model(model_1, num_epochs_1, optimizer_1, loss_fn)

Epoch 50/50: 100%|██████████| 258/258 [00:00<00:00, 808.09it/s]


In [26]:
# Entrenar Modelo 2
train_model(model_2, num_epochs_2, optimizer_2, loss_fn)

Epoch 100/100: 100%|██████████| 258/258 [00:00<00:00, 475.69it/s]


In [27]:
# Entrenar Modelo 3
train_model(model_3, num_epochs_3, optimizer_3, loss_fn)

Epoch 100/100: 100%|██████████| 258/258 [00:00<00:00, 395.89it/s]


## Selección del Mejor Modelo

In [28]:
# Evaluación en Validación para elegir el mejor modelo
print("Resultados de los modelos en Validación:\n")

model_1.eval()
model_2.eval()
model_3.eval()

with torch.no_grad():
    val_pred_1 = model_1(x_val)
    val_loss_1 = loss_fn(val_pred_1, y_val)

    val_pred_2 = model_2(x_val)
    val_loss_2 = loss_fn(val_pred_2, y_val)

    val_pred_3 = model_3(x_val)
    val_loss_3 = loss_fn(val_pred_3, y_val)

print(f"Modelo 1 - MSE en Validación: {val_loss_1.item():.4f}")
print(f"Modelo 2 - MSE en Validación: {val_loss_2.item():.4f}")
print(f"Modelo 3 - MSE en Validación: {val_loss_3.item():.4f}")

Resultados de los modelos en Validación:

Modelo 1 - MSE en Validación: 1.3034
Modelo 2 - MSE en Validación: 0.4139
Modelo 3 - MSE en Validación: 0.4179


## Evaluación del Mejor Modelo

In [30]:
# Evaluar el mejor modelo en Test

best_model = model_2

best_model.eval()

with torch.no_grad():
    test_pred = best_model(x_test)
    test_loss = loss_fn(test_pred, y_test)

print(f"MSE en Prueba (Test): {test_loss.item():.4f}")

MSE en Prueba (Test): 0.4071
